# 03 — Trading Signals and Single-Pair Backtest

**Objective:** convert deviations in the KO/PEP spread into a simple mean-reversion strategy.

I use a rolling z-score to measure how far the spread is from its recent mean.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm

TICKERS = ["KO", "PEP"]
LOOKBACK = 60
ENTRY_Z = 2.0
EXIT_Z = 0.5
COST = 0.0005

prices = yf.download(
    TICKERS,
    start="2018-01-01",
    end="2026-01-01",
    auto_adjust=True,
    progress=False
)["Close"].dropna()

## Build the spread

In [ ]:
model = sm.OLS(
    prices["KO"],
    sm.add_constant(prices["PEP"])
).fit()

alpha = model.params["const"]
beta = model.params["PEP"]

spread = prices["KO"] - (alpha + beta * prices["PEP"])

rolling_mean = spread.rolling(LOOKBACK).mean()
rolling_std = spread.rolling(LOOKBACK).std()
zscore = (spread - rolling_mean) / rolling_std

In [ ]:
zscore.plot(figsize=(10, 4))
plt.axhline(ENTRY_Z, linestyle="--")
plt.axhline(-ENTRY_Z, linestyle="--")
plt.axhline(0)
plt.title("KO/PEP Spread Z-Score")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.grid(alpha=0.3)
plt.show()

## Trading rules

- Enter a long-spread position when the z-score falls below `-2`.
- Enter a short-spread position when the z-score rises above `+2`.
- Exit once the z-score moves back inside `±0.5`.

In [ ]:
position = pd.Series(0.0, index=zscore.index)
current = 0

for i, z in enumerate(zscore):
    if np.isnan(z):
        continue

    if current == 0:
        if z < -ENTRY_Z:
            current = 1
        elif z > ENTRY_Z:
            current = -1
    elif abs(z) < EXIT_Z:
        current = 0

    position.iloc[i] = current

## Backtest

I lag the position by one day so the strategy does not use the same closing price both to create and execute a signal.

In [ ]:
asset_returns = prices.pct_change().fillna(0)

pair_return = (
    asset_returns["KO"] - beta * asset_returns["PEP"]
) / (1 + abs(beta))

lagged_position = position.shift(1).fillna(0)
gross_return = lagged_position * pair_return

turnover = position.diff().abs().fillna(0)
net_return = gross_return - turnover * COST

equity = (1 + net_return).cumprod()

equity.plot(figsize=(10, 4))
plt.title("Single-Pair Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of 1 unit")
plt.grid(alpha=0.3)
plt.show()

print(f"Total return: {equity.iloc[-1] - 1:.2%}")